<div align="center">

# Masterclass Curriculum: Engineering a 2D Adversarial Boss-Slayer AI
## TensorFlow Edition

A polished notebook version of the original blueprint for vision capture, telemetry, action mapping, and DQN training in a 2D boss-fight arena.

</div>

> **Notebook intent:** present the system design clearly, preserve the key engineering details, and make the curriculum easier to scan during study or implementation.

---

## At a Glance

| Section | What it covers |
| --- | --- |
| Step 1 | Frame capture, grayscale compression, and 4-frame temporal stacking |
| Step 2 | Health-mask telemetry, boss-hit signals, and reward shaping |
| Step 3 | Action-space mapping and controller unlatching |
| Step 4 | CNN-based DQN design and Bellman optimization |
| Step 5 | The full runtime loop that binds perception, actuation, and learning |

### System Overview

```text
+------------------+     Raw Pixels     +---------------------------+
|   Game Window    | -----------------> |  Step 1: Vision Pipeline  |
+------------------+                    |  Capture & Frame Stack    |
         ^                              +---------------------------+
         | Virtual Inputs                           |
+------------------+                    +---------------------------+
| Step 3: Actuator |                    | Step 2: Telemetry Engine  |
|   vgamepad       |                    |  Mask & Hit Detection     |
+------------------+                    +---------------------------+
         ^                              +---------------------------+
         | Action Selection                      |
         +---------------------------------- State & Reward Tensors
                                                  |
                                                  v
                                        +---------------------------+
                                        |   Step 4: Policy Graph   |
                                        |      DQN + GradientTape  |
                                        +---------------------------+
```

The key design idea is simple: the model does not just see a frame, it sees motion, danger, and feedback as a structured state tensor.

---

## Master API Dictionary

| Area | Core tools | Why they matter |
| --- | --- | --- |
| Window capture | `pygetwindow.getWindowsWithTitle()`, `mss.mss()`, `sct.grab()` | Locates the game and pulls pixels directly from the screen buffer |
| Vision processing | `cv2.cvtColor()`, `cv2.resize()`, `np.array()` | Converts raw frames into compact grayscale tensors |
| Virtual input | `vgamepad.VX360Gamepad()`, `press_button()`, `release_button()`, `update()` | Turns action indices into real controller events |
| Learning graph | `tf.keras.layers.Conv2D`, `tf.GradientTape()`, `optimizer.apply_gradients()` | Builds and trains the DQN policy network |

### Practical conventions

- Use channel-last tensors shaped like `(height, width, channels)` for TensorFlow.
- Keep frame processing fast: grayscale first, then resize to `84 x 84`.
- Release stale controller inputs before applying the next action.
- Treat reward signals as carefully tuned feedback, not as an afterthought.

---

## Step 1: Vision Pipeline and Temporal Frame Stacking

### Objective
Capture the game window, compress it, and preserve motion by stacking the last four processed frames.

### Why this matters
A single frame is a snapshot. A stacked tensor is a short memory. That short memory is what lets the agent infer direction, velocity, and attack timing.

```text
[Frame t-3]   [Frame t-2]   [Frame t-1]   [Frame t]
   84x84          84x84          84x84        84x84
                                         /
         +------------+------------+----------+
                           |
                           v
                    Stacked State Tensor
                        (84, 84, 4)
```

### Requirements
1. Enforce a fixed sampling interval so the model sees time consistently.
2. Keep a `deque(maxlen=4)` so the oldest frame drops automatically.
3. Stack the queue with `np.stack(..., axis=-1)` to form the model input tensor.

### Clean implementation sequence
- Capture raw pixels from the window bounds.
- Convert BGRA to grayscale.
- Resize to `84 x 84`.
- Append to the frame queue.
- Build the state tensor from the latest four frames.



In [ ]:
import mss
import cv2
import pywinctl
import numpy as np
import time

TARGET_WINDOW = "Hollow Knight"
TOTAL_MASKS = 9
LAST_CHECKED_MASK = 0

sct = mss.MSS()
windows = pywinctl.getWindowsWithTitle(TARGET_WINDOW)

if not windows:
    print("No Target Window Found!")
    exit()
    
win = windows[0]

lower_mask_pink = np.array([210, 205, 222])  # B, G, R
upper_mask_pink = np.array([230, 222, 240])  # B, G, R

def GetMasks(health_frame):
    frame_height, frame_width, _ = health_frame.shape
    mask_width = health_frame.shape[1] // TOTAL_MASKS
    
    current_masks = 0
    for i in range(TOTAL_MASKS):
        start_x = i * mask_width
        end_x = (i + 1) * mask_width
        
        single_mask_box = health_frame[0:frame_height, start_x:end_x]
        
        isolated_color_box = cv2.inRange(single_mask_box, lower_mask_pink, upper_mask_pink)
        
        box_match_pixels = np.sum(isolated_color_box == 255)
        
        if box_match_pixels > 30: 
            current_masks += 1
    return current_masks

def Detections(health_frame, processed_frame):
    global LAST_CHECKED_MASK
    brightness = np.mean(processed_frame)
    current_masks = GetMasks(health_frame=health_frame)
    health_went_down = False
    
    if LAST_CHECKED_MASK < current_masks: # edge case fixes
        LAST_CHECKED_MASK = current_masks
    
    # checks if health went down
    if LAST_CHECKED_MASK > current_masks and brightness < 235:
        amount_down = LAST_CHECKED_MASK - current_masks
        
        if amount_down > 2: # edge case for teleports
            return
        print("Masks went down", current_masks)
        LAST_CHECKED_MASK = current_masks
        health_went_down = True
        time.sleep(1)
    
    # checks if player died
    if current_masks <= 0 and brightness < 230:
        print("Player Died!")
        time.sleep(5)
        LAST_CHECKED_MASK = GetMasks(health_frame=health_frame)
        
        
# Colors for false knight

#MASK 1: main mask for his colors
BOSS_LOWER_COLOR1 = np.array([130, 45, 60])    
BOSS_UPPER_COLOR1 = np.array([150, 120, 180])

# MASK 2: Illuminated/Washed-out Armor (When he is right next to the player's glow)
BOSS_LOWER_COLOR2 = np.array([125, 45, 170]) 
BOSS_UPPER_COLOR2 = np.array([155, 80, 255])

# MASK 3: Mask for when the boss is illuminated
BOSS_LOWER_COLOR3 = np.array([160, 55, 160])  
BOSS_UPPER_COLOR3 = np.array([175, 80, 240])

# MASK 4: mask for the dark armor
BOSS_LOWER_COLOR4 = np.array([128, 70, 15])   
BOSS_UPPER_COLOR4 = np.array([145, 140, 85])

def GetBossBounds(raw_frame):
    hsv_frame = cv2.cvtColor(raw_frame, cv2.COLOR_BGR2HSV)
    
    mask1 = cv2.inRange(hsv_frame, BOSS_LOWER_COLOR1, BOSS_UPPER_COLOR1)
    mask2 = cv2.inRange(hsv_frame, BOSS_LOWER_COLOR2, BOSS_UPPER_COLOR2)
    mask3 = cv2.inRange(hsv_frame, BOSS_LOWER_COLOR3, BOSS_UPPER_COLOR3)
    mask4 = cv2.inRange(hsv_frame, BOSS_LOWER_COLOR4, BOSS_UPPER_COLOR4)
    
    mask = mask1 | mask2 | mask3 | mask4
    
    open_kernel = np.ones((3, 3), np.uint8)
    clean_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, open_kernel)
    
    close_kernel = np.ones((9, 9), np.uint8)
    final_mask = cv2.morphologyEx(clean_mask, cv2.MORPH_CLOSE, close_kernel)
    
    contours, _ = cv2.findContours(final_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return None

    # Find ALL contours that look like parts of the boss (ignoring tiny background specks)
    valid_boss_parts = [cnt for cnt in contours if cv2.contourArea(cnt) > 150]
    
    if not valid_boss_parts:
        return None
    
    # NEW STEP: Stack all pieces together to calculate a single outer boundary box
    all_points = np.vstack(valid_boss_parts)
    x, y, w, h = cv2.boundingRect(all_points)
    
    # Draw the newly stabilized master box
    cv2.rectangle(raw_frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
    
    return (x, y, w, h)


# Track the last frame's white density percentage instead of absolute count
last_frame_white_density = 0.0

def DetectBossHit(raw_frame, boss_bounds):
    global last_frame_white_density, hit_cooldown_frames
    
    if hit_cooldown_frames > 0:
        hit_cooldown_frames -= 1
    
    if boss_bounds is None:
        last_frame_white_density = 0.0
        return False

    x, y, w, h = boss_bounds
    boss_crop = raw_frame[y:y+h, x:x+w]
    
    if boss_crop.size == 0:
        return False
        
    b, g, r, _ = cv2.split(boss_crop)
    
    # Identify bright flashing slash pixels
    bright_pixels = (b > 210) & (g > 210) & (r > 190)
    nail_tint = (b.astype(int) - r.astype(int)) >= 1
    pure_white = (b == 255) & (g == 255) & (r == 255)
    
    hit_mask = (bright_pixels & nail_tint) | pure_white
    current_hit_pixels = np.sum(hit_mask)
    
    total_pixels = boss_crop.shape[0] * boss_crop.shape[1]
    current_density = (current_hit_pixels / total_pixels) * 100
    
    if last_frame_white_density == 0.0:
        last_frame_white_density = current_density
        return False
        
    density_spike = current_density - last_frame_white_density
    last_frame_white_density = current_density
    
    # Tuned to 1.2% to match the new stable, large bounding area
    if density_spike > 1.2 and hit_cooldown_frames == 0:
        hit_cooldown_frames = 3  
        return True
    
    return False
    
    

while True:
    window_dims = {
        "left": win.left,
        "top": win.top,
        "width": win.width,
        "height": win.height
    }
    
    raw_frame = np.array(sct.grab(window_dims))
    
    health_frame = raw_frame[80:120, 245:675, :3] # works when full screen
     
    masks = GetMasks(health_frame=health_frame)
    
    if masks > 0 and LAST_CHECKED_MASK == 0:
        LAST_CHECKED_MASK = masks

    processed_frame = cv2.resize(raw_frame, (128, 90), interpolation=cv2.INTER_LINEAR)
    processed_frame = cv2.cvtColor(processed_frame, cv2.COLOR_BGRA2GRAY)
    
    Detections(health_frame=health_frame, processed_frame=processed_frame)
    
    boss_box = GetBossBounds(raw_frame=raw_frame)
    if DetectBossHit(raw_frame=raw_frame, boss_bounds=boss_box):
        print("Hit Boss!")
    
   #cv2.imshow("Raw - altered", raw_frame)
    cv2.imshow("Display", processed_frame)
    cv2.imshow("HealthBar", health_frame)

    # print(f"HEALTH MASKS DETECTED: {masks} / {TOTAL_MASKS} {LAST_CHECKED_MASK}", end="\r")
    
    if cv2.waitKey(1) & 0xFF == ord("q"):
        cv2.destroyAllWindows()
        break


Masks went down 8
Hit Boss!
Hit Boss!
Hit Boss!
Hit Boss!
Hit Boss!
Hit Boss!
Hit Boss!
Hit Boss!
Hit Boss!
Hit Boss!
Hit Boss!
Hit Boss!
Hit Boss!
Hit Boss!
Hit Boss!
Hit Boss!
Hit Boss!
Masks went down 8
Hit Boss!
Hit Boss!
Hit Boss!


KeyboardInterrupt: 

---

## Step 2: Telemetry and Reward Engineering

### Core idea
Because the game does not expose internal state directly, reward must be inferred from visual cues.

### Player health tracking
- Crop the UI region where health masks appear.
- Threshold the crop for bright pixels.
- Compare the current white-pixel count to the previous step.
- Treat a significant drop as damage taken.

### Boss-hit detection
- Watch for sudden bright particle bursts around the combat zone.
- Use repeated identical frames as a freeze-frame hint for landed hits.
- Combine both signals for a more stable offensive reward.

### Reward matrix

| Telemetry trigger | Reward | Purpose |
| --- | --- | --- |
| Mask broken (player hit) | `-50.0` | Strongly penalizes failed dodges |
| Boss particle burst detected | `+20.0` | Rewards successful offense |
| Step survival incentive | `+0.1` | Encourages steady movement and survival |
| Stagnation penalty | `-0.05` | Prevents passive turtling |

### Design note
The reward scale should discourage standing still more than it encourages reckless aggression.

---

## Step 3: Actuator Mapping

### Action space
Each network output is an integer index that maps to a concrete controller behavior.

```python
ACTION_SPACE = {
    0: "NEUTRAL",
    1: "MOVE_LEFT",
    2: "MOVE_RIGHT",
    3: "JUMP",
    4: "ATTACK",
    5: "DASH",
    6: "MOVE_LEFT_AND_ATTACK",
    7: "MOVE_RIGHT_AND_ATTACK",
}
```

### Unlatching requirement
Before applying a new action, release the buttons from the previous action so inputs do not remain stuck across steps.

### Execution pattern
1. Clear the previous input state.
2. Choose the next action index.
3. Apply the corresponding joystick or button command.
4. Call `gamepad.update()` once to flush the entire state.

---

In [ ]:
ACTION_SPACE = {
    0: "NEUTRAL",
    1: "MOVE_LEFT",
    2: "MOVE_RIGHT",
    3: "JUMP",
    4: "ATTACK",
    5: "DASH",
    6: "MOVE_LEFT_AND_ATTACK",
    7: "MOVE_RIGHT_AND_ATTACK",
}

def reset_controller(gamepad):
    """Release stale inputs before the next action is applied."""
    gamepad.left_joystick_float(x_value_float=0.0, y_value_float=0.0)
    gamepad.release_button(vgamepad.XUSB_BUTTON.XUSB_GAMEPAD_A)
    gamepad.release_button(vgamepad.XUSB_BUTTON.XUSB_GAMEPAD_X)
    gamepad.release_button(vgamepad.XUSB_BUTTON.XUSB_GAMEPAD_RIGHT_SHOULDER)


## Step 4: The Network Brain and Gradient Engine

### Model structure
Your DQN should process the stacked `(84, 84, 4)` state tensor through a compact CNN pipeline:

1. `Conv2D` for broad spatial features.
2. `Conv2D` for tighter movement cues.
3. `Flatten` to collapse the feature maps.
4. `Dense(512, relu)` for nonlinear policy reasoning.
5. Linear output layer sized to the action space.

### Stabilization strategy
- The policy network learns every step.
- The target network stays frozen between syncs.
- Copy weights periodically, such as every 1,000 steps.

### Optimization loop
Wrap loss computation inside `tf.GradientTape()`, compute the Bellman target, and apply gradients with `optimizer.apply_gradients(...)`.

```text
policy_net -> Q-values -> action choice
replay buffer -> batch sampling -> Bellman target
target_net -> frozen bootstrap values
gradient tape -> loss -> optimizer step
```

---

In [ ]:
import tensorflow as tf

class DQN(tf.keras.Model):
    def __init__(self, action_count):
        super().__init__()
        self.conv1 = tf.keras.layers.Conv2D(32, 8, strides=4, activation="relu")
        self.conv2 = tf.keras.layers.Conv2D(64, 4, strides=2, activation="relu")
        self.flatten = tf.keras.layers.Flatten()
        self.hidden = tf.keras.layers.Dense(512, activation="relu")
        self.output_layer = tf.keras.layers.Dense(action_count, activation=None)

    def call(self, inputs, training=False):
        x = self.conv1(inputs)
        x = self.conv2(x)
        x = self.flatten(x)
        x = self.hidden(x)
        return self.output_layer(x)


## Step 5: The Master Optimization Loop

### Runtime blueprint
1. Initialize the window handle, screen capture, virtual controller, policy network, target network, and replay buffer.
2. Pause briefly so the game window can receive focus.
3. Enter the real-time loop and repeat the perception, telemetry, action, and learning cycle.
4. Sync the target network on a fixed schedule.
5. On exit, release controller inputs and shut everything down cleanly.

```text
while running:
    capture frame
    compute telemetry
    stack frames
    pick action
    send controller input
    store transition
    learn from replay
    sync target network periodically
```

### Closing note
> Start by validating the telemetry hook first. If the pixel-density signal is wrong, every later layer of the system will train on noise.

---

### Build order that keeps risk low
- Get the health-mask counter working first.
- Add frame preprocessing and the 4-frame stack next.
- Wire action translation only after telemetry is stable.
- Add the DQN and replay buffer last.
